In [ ]:
import sys
from pathlib import Path

for root in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (root / "src" / "willow").exists():
        sys.path.insert(0, str(root / "src"))
        break
for name in list(sys.modules):
    if name == "willow" or name.startswith("willow."):
        del sys.modules[name]

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from willow import models as md, metrics as mt, interpret as it
from willow.data import ROOT, load_panel
from willow.features import analytic_windows, build_windows, make_X, n_declined
from willow.schema import ITEM_LABELS

pd.set_option("display.max_columns", 40, "display.width", 160, "display.precision", 3)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

OUTCOME = "y_any"        # newly receives help with any of the 7 activities at t+1
FEATURES = "change"      # status + extra measures + change since last year (main model)
SPLIT_ROUND = 11         # SAME as 03_model: train outcomes up to 2021, test windows from 2021 on
N_ITER = 15              # random-search candidates per tree model (lower to ~6 for a quick run)
OUT = os.path.join(ROOT, "results")
os.makedirs(OUT, exist_ok=True)

In [ ]:
a = analytic_windows(build_windows(load_panel()), OUTCOME)
train, test = md.person_time_split(a, SPLIT_ROUND)
assert set(train.person_id).isdisjoint(test.person_id)
pd.DataFrame({"train": {"windows": len(train), "people": train.person_id.nunique(), "events": int(train[OUTCOME].sum())},
              "test": {"windows": len(test), "people": test.person_id.nunique(), "events": int(test[OUTCOME].sum())}}).T

In [ ]:
X_train, y_train = make_X(train, FEATURES), train[OUTCOME].values
X_test, y_test = make_X(test, FEATURES), test[OUTCOME].values
folds = md.grouped_folds(train, OUTCOME)
print(f"{X_train.shape[1]} features | {len(folds)} grouped folds")

In [ ]:
def run(name, tune):
    t0 = time.time()
    cv_table, best, oof = tune()
    model = md.logistic(best) if name == "lr_change" else md.TREE_MODELS[name.split("_")[0]][0](**best)
    model.fit(X_train, y_train)
    platt = md.platt_fit(oof, y_train)
    threshold = mt.alert_threshold(md.platt_apply(platt, oof))
    p_test = md.platt_apply(platt, model.predict_proba(X_test)[:, 1])
    ci = mt.bootstrap_ci(y_test, p_test, test.person_id.values, threshold)
    print(f"{name:10s} test AUROC {ci.loc['auroc', 'estimate']:.3f}  ({time.time() - t0:.0f}s)  best={best}")
    return dict(name=name, model=model, best=best, cv=cv_table, oof=oof, platt=platt,
                threshold=threshold, p_test=p_test, ci=ci)

R = {}
R["lr_change"] = run("lr_change", lambda: md.tune_logistic(X_train, y_train, folds))
R["rf_change"] = run("rf_change", lambda: md.tune_tree("rf", X_train, y_train, folds, n_iter=N_ITER))
R["xgb_change"] = run("xgb_change", lambda: md.tune_tree("xgb", X_train, y_train, folds, n_iter=N_ITER))

In [ ]:
LABELS = {"lr_status": "Logistic regression on status only (baseline)",
          "lr_change": "Logistic regression with change features",
          "rf_change": "Random forest", "xgb_change": "XGBoost"}

cis = {}
base_path = os.path.join(OUT, "model_lr_status_metrics.csv")
if os.path.exists(base_path):
    cis["lr_status"] = pd.read_csv(base_path, index_col=0)[["estimate", "ci_low", "ci_high"]]
cis.update({k: v["ci"] for k, v in R.items()})

def fmt(ci, m, kind, with_ci=True):
    e, lo, hi = ci.loc[m, ["estimate", "ci_low", "ci_high"]]
    f = (lambda v: f"{100 * v:.0f}%") if kind == "pct" else (lambda v: f"{v:.3f}" if kind == "3" else f"{v:.2f}")
    return f"{f(e)} [{f(lo)}–{f(hi)}]" if with_ci else f(e)

COLS = [("AUROC", "auroc", "3"), ("AUPRC", "auprc", "2"), ("Observed/expected", "obs_exp", "2"),
        ("Calibration slope", "cal_slope", "2"), ("Top 20% alert: sensitivity", "sensitivity", "pct"),
        ("Top 20% alert: precision", "precision", "pct")]
table = pd.DataFrame({col: {LABELS[k]: fmt(ci, m, kind) for k, ci in cis.items()} for col, m, kind in COLS})
slide = pd.DataFrame({col: {LABELS[k]: fmt(ci, m, kind, False) for k, ci in cis.items()} for col, m, kind in COLS})
print("For the slide (estimates only):")
display(slide)
table

In [ ]:
def paired_auroc_diff(p_a, p_b, n=1000, seed=5600):
    ids, inv = np.unique(test.person_id.values, return_inverse=True)
    rows_of = [np.flatnonzero(inv == i) for i in range(len(ids))]
    rng, d = np.random.default_rng(seed), []
    for _ in range(n):
        idx = np.concatenate([rows_of[i] for i in rng.integers(0, len(ids), len(ids))])
        if 0 < y_test[idx].sum() < len(idx):
            d.append(roc_auc_score(y_test[idx], p_a[idx]) - roc_auc_score(y_test[idx], p_b[idx]))
    lo, hi = np.quantile(d, [0.025, 0.975])
    return roc_auc_score(y_test, p_a) - roc_auc_score(y_test, p_b), lo, hi

diffs = pd.DataFrame([dict(zip(["AUROC difference", "ci_low", "ci_high"],
                               paired_auroc_diff(R[k]["p_test"], R["lr_change"]["p_test"])), comparison=f"{k} − lr_change")
                      for k in ["rf_change", "xgb_change"]]).set_index("comparison").round(3)
diffs

In [ ]:
fig_cal, ax = plt.subplots(figsize=(5, 5))
lim = 0
for (k, r), col in zip(R.items(), ["#2a78d6", "#1b9e77", "#d95f02"]):
    cal = mt.calibration_table(y_test, r["p_test"])
    lim = max(lim, cal.predicted.max(), cal.observed.max())
    ax.plot(cal.predicted, cal.observed, "o-", ms=5, color=col, label=LABELS[k].replace(" with change features", " + change"))
lim *= 1.1
ax.plot([0, lim], [0, lim], ls="--", color="#999", lw=1)
ax.set(xlim=(0, lim), ylim=(0, lim), xlabel="Predicted risk", ylabel="Observed rate", title="Calibration on the test set (deciles)")
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}"); ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.legend(frameon=False, fontsize=8); plt.tight_layout()

In [ ]:
def nice(f):
    base = f.split("_", 1)[1] if f.startswith(("cur_", "d_", "new_")) else f
    return ITEM_LABELS.get(base, f) + (" (change)" if f.startswith(("d_", "new_")) else "")

shap_out = {}
for k in ["xgb_change", "rf_change"]:
    sv, Xs = it.tree_shap(R[k]["model"], X_train)
    imp = it.shap_importance(sv, Xs)
    imp.insert(1, "label", [nice(f) for f in imp.feature])
    shap_out[k] = imp
shap_out["xgb_change"].head(15)

In [ ]:
grouped = it.shap_by_change_group(shap_out["xgb_change"])
fig_shap, ax = plt.subplots(figsize=(6.5, 4.5))
g = grouped.head(12).iloc[::-1]
ax.barh(g["group"], g["mean_abs_shap"], color="#2a78d6")
ax.set(xlabel="Mean |SHAP| (XGBoost)", title="Which kinds of change drive predicted risk")
plt.tight_layout()
grouped.head(12)

In [ ]:
best_tree = max(["rf_change", "xgb_change"], key=lambda k: R[k]["ci"].loc["auroc", "estimate"])
r = R[best_tree]
p_train = md.platt_apply(r["platt"], r["oof"])
print("best tree model:", best_tree)
display(it.alert_levels(y_test, r["p_test"], p_train))
it.error_profiles(test, r["p_test"], OUTCOME, r["threshold"], n_declined(test))

In [ ]:
for k, r in R.items():
    r["ci"].assign(model=k, threshold=r["threshold"], best=str(r["best"])).to_csv(os.path.join(OUT, f"model_{k}_metrics.csv"))
    r["cv"].to_csv(os.path.join(OUT, f"model_{k}_cv.csv"), index=False)
    mt.calibration_table(y_test, r["p_test"]).to_csv(os.path.join(OUT, f"model_{k}_calibration.csv"), index=False)
for k, imp in shap_out.items():
    imp.to_csv(os.path.join(OUT, f"model_{k}_shap.csv"), index=False)
grouped.to_csv(os.path.join(OUT, "model_xgb_change_shap_groups.csv"), index=False)
table.to_csv(os.path.join(OUT, "midpoint_model_table.csv"))
diffs.to_csv(os.path.join(OUT, "midpoint_auroc_differences.csv"))
fig_cal.savefig(os.path.join(OUT, "fig_change_models_calibration.png"), dpi=200, bbox_inches="tight")
fig_shap.savefig(os.path.join(OUT, "fig_xgb_change_shap_groups.png"), dpi=200, bbox_inches="tight")
print("saved results/model_{lr,rf,xgb}_change_*.csv, midpoint_model_table.csv and figures")